# Atelier Préparation de Données Images

## Structure du projet

In [ ]:
from pathlib import Path

RAW_DIR = Path("../data/raw")
CLEANED_DIR = Path("../data/cleaned")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

TAILLE_MIN = 64            
TAILLE_CIBLE = (224, 224)  

audit = {}  # dictionnaire qui accumulera les métriques pour reports/audit_images.csv


## Partie 1 – Exploration du dataset

*Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l'écart-type de ses pixels, son nombre de canaux et sa taille (NB : prendre en charge aussi les fichiers corrompus).*

In [3]:
import numpy as np
import pandas as pd
from PIL import Image

def explorer_image(chemin_image: Path, classe: str) -> dict:
    """Récupère les métadonnées d'une image. Renvoie un dict même si l'image est corrompue."""
    infos = {
        "nom": chemin_image.name,
        "classe": classe,
        "chemin": str(chemin_image),
        "format": None,
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "ecart_type_pixels": None,
        "nb_canaux": None,
        "taille_octets": chemin_image.stat().st_size,
        "corrompue": False,
    }
    try:
        with Image.open(chemin_image) as img:
            img.verify()

        with Image.open(chemin_image) as img:
            infos["format"] = img.format
            infos["mode"] = img.mode
            infos["largeur"], infos["hauteur"] = img.size

            arr = np.array(img)
            infos["ecart_type_pixels"] = float(arr.std())
            infos["nb_canaux"] = 1 if arr.ndim == 2 else arr.shape[2]

    except Exception:
        infos["corrompue"] = True

    return infos


def explorer_dataset(raw_dir: Path) -> pd.DataFrame:
    lignes = []
    for classe in CLASSES:
        dossier_classe = raw_dir / classe
        if not dossier_classe.exists():
            continue
        for fichier in sorted(dossier_classe.iterdir()):
            if fichier.is_file():
                lignes.append(explorer_image(fichier, classe))
    return pd.DataFrame(lignes)


df = explorer_dataset(RAW_DIR)
audit["total_images"] = len(df)
print(f"Total d\'images explorées : {len(df)}")
df.head()


Total d'images explorées : 1032


,nom,classe,chemin,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
0,cardboard1.jpg,cardboard,../data/raw/cardboard/cardboard1.jpg,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False
1,cardboard10.jpg,cardboard,../data/raw/cardboard/cardboard10.jpg,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False
2,cardboard100.jpg,cardboard,../data/raw/cardboard/cardboard100.jpg,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False
3,cardboard101.jpg,cardboard,../data/raw/cardboard/cardboard101.jpg,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False
4,cardboard102.jpg,cardboard,../data/raw/cardboard/cardboard102.jpg,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False


## Partie 2 – Détecter les images corrompues

*Écrire et se servir d'une fonction qui détecte une image corrompue.*

In [4]:
from PIL import Image

def est_corrompue(chemin_image: Path) -> bool:
    """Renvoie True si l'image ne peut pas être ouverte/vérifiée par PIL."""
    try:
        with Image.open(chemin_image) as img:
            img.verify()
        return False
    except Exception:
        return True


images_corrompues = df[df["corrompue"] == True]
audit["images_corrompues"] = len(images_corrompues)
print(f"Images corrompues : {len(images_corrompues)}")
images_corrompues[["nom", "classe", "chemin"]]


Images corrompues : 6


,nom,classe,chemin
147,cardboard83.jpg,cardboard,../data/raw/cardboard/cardboard83.jpg
326,glass74.jpg,glass,../data/raw/glass/glass74.jpg
446,metal48.jpg,metal,../data/raw/metal/metal48.jpg
633,paper213.jpg,paper,../data/raw/paper/paper213.jpg
791,plastic13.jpg,plastic,../data/raw/plastic/plastic13.jpg
1004,trash3.jpg,trash,../data/raw/trash/trash3.jpg


## Partie 3 – Détecter les images vides
Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image entièrement blanche ou image dont les pixels présentent très peu de variation.

In [5]:
import numpy as np
from PIL import Image

SEUIL_ECART_TYPE = 5.0  # en dessous de ce seuil, l'image est jugée quasi vide

def est_image_vide(chemin_image: Path, seuil: float = SEUIL_ECART_TYPE) -> bool:
    """Détecte une image entièrement noire/blanche ou à très faible variation de pixels."""
    try:
        with Image.open(chemin_image) as img:
            arr = np.array(img.convert("L"))
    except Exception:
        return False  # les corrompues sont déjà traitées en Partie 2

    if arr.std() < seuil:
        return True
    if np.all(arr == 0) or np.all(arr == 255):
        return True
    return False


df_valides = df[df["corrompue"] == False].copy()
df_valides["vide"] = df_valides["chemin"].apply(lambda p: est_image_vide(Path(p)))

images_vides = df_valides[df_valides["vide"] == True]
audit["images_quasi_vides"] = len(images_vides)
print(f"Images quasi vides : {len(images_vides)}")
images_vides[["nom", "classe", "ecart_type_pixels"]]


Images quasi vides : 4


,nom,classe,ecart_type_pixels
167,image-blanche-512x384.jpg,cardboard,1.572536
355,image-noire-512x384.png,glass,110.418239
357,image-blanche-512x384.jpg,metal,1.572536
358,image-noire-512x384.png,metal,110.418239


## Partie 4 – Détecter les différences de résolution
### 1) Déterminer la résolution minimale, la résolution maximale, les résolutions les plus fréquentes et le nombre d'images par résolution.

In [6]:
import pandas as pd

df_valides["resolution"] = list(zip(df_valides["largeur"], df_valides["hauteur"]))

res_min = (df_valides["largeur"].min(), df_valides["hauteur"].min())
res_max = (df_valides["largeur"].max(), df_valides["hauteur"].max())
compte_resolutions = df_valides["resolution"].value_counts()

print(f"Résolution minimale : {res_min}")
print(f"Résolution maximale : {res_max}")
print("\nRésolutions les plus fréquentes :")
print(compte_resolutions.head(10))


Résolution minimale : (np.float64(32.0), np.float64(32.0))
Résolution maximale : (np.float64(512.0), np.float64(384.0))

Résolutions les plus fréquentes :
resolution
(512.0, 384.0)    1013
(32.0, 32.0)         5
(48.0, 32.0)         4
(40.0, 40.0)         4
Name: count, dtype: int64
